In [1]:
import tensorflow as tf
from tensorflow.keras.models import load_model
import pickle
import pandas as pd
import numpy as np

In [3]:
##load trained model, scaler pickle, onehot encoder and label encoder
model=load_model('model.h5')

with open('onehotencoder_geo.pkl', 'rb') as file:
    label_encoder_geo=pickle.load(file)

with open('label_encoder_gender.pkl', 'rb') as file:
    label_encoder_gender=pickle.load(file)    

with open('scaler.pkl', 'rb') as file:
    scaler = pickle.load(file)    

In [13]:
#Example input data
input_data= {
    'CreditScore': 600,
    'Geography': 'Germany',
    'Gender': 'Female',
    'Age': 45,
    'Tenure': 10,
    'Balance': 80000,
    'NumOfProducts': 4,
    'HasCrCard': 1,
    'IsActiveMember': 1,
    'EstimatedSalary': 75000
}

In [14]:
#One-Hot encoding Geography
geo_encoded=label_encoder_geo.transform([[input_data['Geography']]]).toarray()
geo_encoded_df=pd.DataFrame(geo_encoded, columns=label_encoder_geo.get_feature_names_out(['Geography']))
geo_encoded_df

/Users/sandeshsaidapur/ML Learning projects/ANNClassification/venv/lib/python3.11/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but OneHotEncoder was fitted with feature names
  warnings.warn(


,Geography_France,Geography_Germany,Geography_Spain
0,0.0,1.0,0.0


In [15]:
input_df=pd.DataFrame([input_data])
input_df

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary
0,600,Germany,Female,45,10,80000,4,1,1,75000


In [16]:
#Encoding categorical variables
input_df['Gender']=label_encoder_gender.transform(input_df['Gender'])
input_df

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary
0,600,Germany,0,45,10,80000,4,1,1,75000


In [19]:
#Combining one-hot encoded columns with input data
input_df=pd.concat([input_df.drop("Geography", axis=1), geo_encoded_df], axis=1)
input_df

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Geography_France,Geography_Germany,Geography_Spain
0,600,0,45,10,80000,4,1,1,75000,0.0,1.0,0.0


In [20]:
##Scaling input data
input_scaled=scaler.transform(input_df)
input_scaled

array([[-0.53598516, -1.09499335,  0.58015577,  1.7337772 ,  0.0624086 ,
         4.25868381,  0.64920267,  0.97481699, -0.44216545, -0.99850112,
         1.72572313, -0.57638802]])

In [21]:
##Predict churn
prediction=model.predict(input_scaled)
prediction

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 63ms/step


array([[0.9999972]], dtype=float32)

In [22]:
prediction_probability=prediction[0][0]
prediction_probability

np.float32(0.9999972)

In [23]:
if prediction_probability > 0.5:
    print("The customer is likely to churn")
else:
    print("The customer is not likely to churn")    

The customer is likely to churn
